In [1]:
%pip install -q langchain-openai langchain-core requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
from typing import Annotated
from langchain_core.tools import tool, InjectedToolArg

In [13]:
# create the tool

@tool
def get_conversion_factor(base_currency: str, target_currency:str) -> float:
    """ This function fetches the currency conversion factor between a base currency and 
    a target currency """
    url = f'https://v6.exchangerate-api.com/v6/74229d69cfe6706b5d983e0c/pair/{base_currency}/{target_currency}'

    response = requests.get(url)
    return response.json()



In [14]:
@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate

In [15]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [16]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1782518402,
 'time_last_update_utc': 'Sat, 27 Jun 2026 00:00:02 +0000',
 'time_next_update_unix': 1782604802,
 'time_next_update_utc': 'Sun, 28 Jun 2026 00:00:02 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 94.4541}

In [17]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

851.5999999999999

In [18]:
# tool binding
llm = ChatOpenAI()

In [19]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [26]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 100 inr to usd')]

In [27]:
ai_message = llm_with_tools.invoke(messages)

In [28]:
messages.append(ai_message)

In [29]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': 'call_5OYsnYiPjrIdVBKzZDvgcxlm',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 100},
  'id': 'call_GZ2AFXJP7OOFkpFGdJIA47vR',
  'type': 'tool_call'}]

In [30]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)

In [31]:
llm_with_tools.invoke(messages).content

'The conversion factor between INR and USD is 0.01059. \n\nBased on this conversion factor, 100 INR is equal to approximately 1.059 USD.'